In [ ]:
import os
import json
from config import GAMES, DEFAULT_MAX_STEPS, DIFFICULTY_MAX_STEPS_MAP, NUM_EPISODES, LLM_BACKEND, LLM_MODEL, LLM_TEMPERATURE
from src.retrieval import build_graph
from src.utils import init_game_env, get_llm
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import HumanMessage
from langgraph.store.memory import InMemoryStore
import time

In [2]:
def run_single_game(game_name: str, game_path: str, difficulty:str, llm, max_steps: int, num_episodes: int = 1):
    env = init_game_env(game_path, max_steps=max_steps)
    episodic_store = InMemoryStore(
    index={
        "dims": 1536,
        "embed": "openai:text-embedding-3-small", # Model for vectorization
    }
)

    config = RunnableConfig(recursion_limit=4*max_steps, configurable={"episodic_store": episodic_store})
    all_results = []
    game_log_dir = os.path.join("logs", "retrieval", game_name)
    os.makedirs(game_log_dir, exist_ok=True)

    print(f"=============== Starting game: {game_name} [{difficulty}, {max_steps}] with {num_episodes} episodes ===============")

    for episode in range(num_episodes):
        # Graph with its scrapthpad immidiete message history should 
        # be reset per episode, but long term memory should be persistance 
        # across single game session
        graph = build_graph(llm=llm, env=env)
        print(f"\t=============== Running episode {episode + 1}/{num_episodes} ")
        obs, infos = env.reset()
        state = {

            "messages": [],
            "step": 1,
            "trace": [],
            "current_obs": obs,
            "score_count": 0,
            "done": False,
            "infos": infos,
            "episode_num": episode + 1,
        }


        start_time = time.time()
        output = graph.invoke(state, config=config)
        duration = time.time() - start_time

        trace = output.get("trace", [])
        total_tokens = sum(t.get("tokens_total", 0) for t in trace)
        total_prompt_tokens = sum(t.get("tokens_prompt", 0) for t in trace)
        total_completion_tokens = sum(t.get("tokens_completion", 0) for t in trace)

        
        steps_taken = output.get("step", 0)
        
        episode_result = {
            "episode": episode,
            "game": game_name,
            "difficulty": difficulty,
            "model_temperature": LLM_TEMPERATURE,
            "max_steps": max_steps,
            "agent": "retrieval",
            "duration_seconds": round(duration, 2),
            "moves":  output.get("infos", 0).get("moves", 0),
            "max_score": output.get("infos", 0).get("max_score", 0),
            "won": output.get("infos", 0).get("won", 0),
            "lost": output.get("infos", 0).get("lost", 0),
            "score": output.get("score_count", 0),
            "steps_taken": steps_taken-1,
            "trace": output.get("trace", []),
            "total_tokens": total_tokens,
            "prompt_tokens": total_prompt_tokens,
            "completion_tokens": total_completion_tokens,
            "avg_tokens_per_step": round(total_tokens / steps_taken, 2) if steps_taken else 0,
            "trace": trace
        }

        # Save per-episode file
        episode_path = os.path.join(game_log_dir, f"episode_{episode}.json")
        with open(episode_path, "w") as f:
            json.dump(episode_result, f, indent=2)

        all_results.append(episode_result)
        print(f"\t=============== Finished episode {episode + 1}/{num_episodes}")
        print(f"""\tWon: {episode_result['won']}\n\tLost: {episode_result['lost']}\n\tScore: {episode_result['score']}\n\tMax Score: {episode_result["max_score"]}\n\tSteps Taken: {episode_result['steps_taken']}\n\tDuration: {episode_result['duration_seconds']} seconds""")

    print(f"All episodes completed for game: {game_name} [{difficulty}, {max_steps}].")
    print(f"Num won: {sum(1 for r in all_results if r['won'])}\nNum lost: {sum(1 for r in all_results if r['lost'])}")
    return all_results

def run_all_games():
    summary = []
    llm = get_llm(backend=LLM_BACKEND, model=LLM_MODEL, temperature=LLM_TEMPERATURE)

    for game in GAMES:
        name = game["name"]
        path = game["path"]
        difficulty = game["difficulty"]
        max_steps = DIFFICULTY_MAX_STEPS_MAP.get(difficulty, DEFAULT_MAX_STEPS)

        episodes = run_single_game(game_name=name, game_path=path, llm=llm, difficulty=difficulty, max_steps=max_steps, num_episodes=NUM_EPISODES)
        for ep_result in episodes:
            summary.append({"game": name, **ep_result})

    # Save summary
    os.makedirs("logs/retrieval", exist_ok=True)
    with open("logs/retrieval/summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("Finished all games. Summary saved to logs/retrieval/summary.json")


In [3]:
if __name__ == "__main__":
    run_all_games()

=============== Starting game: tw-cooking-recipe1+take1+cook-xW11fkvmtmZUxng [easy, 20] with 4 episodes ===============
	=============== Running episode 1/4 
	=============== Finished episode 1/4
	Won: True
	Lost: False
	Score: 6
	Max Score: 3
	Steps Taken: 4
	Duration: 10.1 seconds
	=============== Running episode 2/4 
	=============== Finished episode 2/4
	Won: True
	Lost: False
	Score: 6
	Max Score: 3
	Steps Taken: 3
	Duration: 7.95 seconds
	=============== Running episode 3/4 
	=============== Finished episode 3/4
	Won: True
	Lost: False
	Score: 6
	Max Score: 3
	Steps Taken: 4
	Duration: 8.55 seconds
	=============== Running episode 4/4 
	=============== Finished episode 4/4
	Won: True
	Lost: False
	Score: 6
	Max Score: 3
	Steps Taken: 4
	Duration: 8.97 seconds
All episodes completed for game: tw-cooking-recipe1+take1+cook-xW11fkvmtmZUxng [easy, 20].
Num won: 4
Num lost: 0
=============== Starting game: tw-cooking-recipe1+take1+cut-W5NyHaWWHY2nh3Vj [easy, 20] with 4 episodes ======